In [ ]:
import os
import random
from itertools import product
from time import perf_counter

import numpy as np
import polars as pl
from aeon.classification.base import BaseClassifier
from aeon.classification.feature_based import (
    Catch22Classifier,
)
from aeon.datasets.tsc_datasets import univariate
from joblib import Parallel, delayed
from sklearn.base import clone
from sklearn.ensemble import (
    RandomForestClassifier,
)
from sklearn.metrics import accuracy_score

from autotsc import utils

In [ ]:
class CrossValidationWrapper(BaseClassifier):
    def __init__(self, model):
        self.model = model
        self.trained_models_ = []
        super().__init__()

    def _fit(self, X, y):
        pass

    def _predict(self, X):
        all_proba = [m.predict_proba(X) for m in self.trained_models_]
        mean_proba = np.mean(all_proba, axis=0)
        return self.classes_[np.argmax(mean_proba, axis=1)]

    def _fit_predict_proba(self, X, y, cv_splits):
        self.fit(X, y)
        oof_proba = np.zeros((len(y), len(np.unique(y))))

        for train_idx, val_idx in cv_splits:
            model = clone(self.model)
            model.fit(X[train_idx], y[train_idx])
            self.trained_models_.append(model)
            oof_proba[val_idx] = model.predict_proba(X[val_idx])
        return oof_proba

    def _fit_predict(self, X, y, cv_splits):
        proba = self.fit_predict_proba(X=X, y=y, cv_splits=cv_splits)
        return self.classes_[np.argmax(proba, axis=1)]

In [ ]:
def get_model(name, size=None, n_jobs=4, random_state=42):
    match (size, name.lower()):
        case ("xs", "catch22"):
            return CrossValidationWrapper(
                Catch22Classifier(
                    n_jobs=n_jobs,
                    random_state=random_state,
                    estimator=RandomForestClassifier(n_estimators=10, n_jobs=n_jobs),
                )
            )
        case ("s", "catch22"):
            return CrossValidationWrapper(
                Catch22Classifier(
                    n_jobs=n_jobs,
                    random_state=random_state,
                    estimator=RandomForestClassifier(n_estimators=70, n_jobs=n_jobs),
                )
            )
        case ("m", "catch22"):
            return CrossValidationWrapper(
                Catch22Classifier(
                    n_jobs=n_jobs,
                    random_state=random_state,
                    estimator=RandomForestClassifier(n_estimators=200, n_jobs=n_jobs),
                )
            )
        case ("l", "catch22"):
            return CrossValidationWrapper(
                Catch22Classifier(
                    n_jobs=n_jobs,
                    random_state=random_state,
                    estimator=RandomForestClassifier(n_estimators=500, n_jobs=n_jobs),
                )
            )
        case ("xl", "catch22"):
            return CrossValidationWrapper(
                Catch22Classifier(
                    n_jobs=n_jobs,
                    random_state=random_state,
                    estimator=RandomForestClassifier(n_estimators=1000, n_jobs=n_jobs),
                )
            )
        case (None, "catch22"):
            return Catch22Classifier(n_jobs=n_jobs, random_state=random_state)
        # case ('xs', RocketClassifier)
        case _:
            return "Invalid size"

In [ ]:
write_dir = "experiments/automl_ca_vs_time_correlation"
os.makedirs(write_dir, exist_ok=True)

datasets = list(univariate)
random.shuffle(datasets)

rng = random.Random()
rng.shuffle(datasets)
datasets = datasets
print(datasets)
model_types = ["catch22"]

n_jobs = 12
n_runs = 10

In [ ]:
def process(ds, run, model_name, size):
    try:
        print("Started: ", ds, run, model_name, size)
        m = get_model(name=model_name, size=size, n_jobs=n_jobs, random_state=run)
        full_model_name = f"{size}-{model_name}" if size is not None else model_name
        stats = {
            "dataset": ds,
            "run": run,
            "model": full_model_name,
        }

        hash_val = pl.DataFrame([stats]).hash_rows(seed=42, seed_1=1, seed_2=2, seed_3=3).item()
        file = f"{write_dir}/{hash_val}.parquet"

        if os.path.exists(file):
            return
        else:
            pass

        X_train, y_train, X_test, y_test = utils.load_dataset(ds)
        folds = get_folds(X_train, y_train, n_splits=10)

        fit_predict_cv_exists = hasattr(m, "fit_predict") and callable(getattr(m, "fit_predict"))
        val_acc = None
        start_time = perf_counter()
        if fit_predict_cv_exists:
            y_pred = m.fit_predict(X=X_train, y=y_train, cv_splits=folds)
            val_acc = accuracy_score(y_train, y_pred)
        else:
            m.fit(X_train, y_train)
            y_pred = m.predict(X_train)

        end_time = perf_counter()
        test_acc = accuracy_score(y_test, m.predict(X_test))
        train_time = end_time - start_time

        stats["test_accuracy"] = test_acc
        stats["training_time"] = train_time
        stats["val_accuracy"] = val_acc

        df_stat = pl.DataFrame([stats])
        df_stat.write_parquet(file)

        print(f"Completed: Dataset={ds}, Run={run}, Model={full_model_name}")
    except Exception as e:
        print(f"Error processing Dataset={ds}, Run={run}, Model={full_model_name}: {e}")
        pass


jobs = []
all_combinations = list(product(datasets, range(n_runs), model_types))
for ds, run, model_name in all_combinations:
    for size in [None, "xs", "s", "m", "l", "xl"]:
        jobs.append(delayed(process)(ds, run, model_name, size))
r = Parallel(n_jobs=16, backend="threading")(jobs)

In [ ]:
df = pl.read_parquet(f"{write_dir}/*.parquet")  # .filter(pl.col("dataset").is_in(datasets))
df

In [ ]:
stats_pivot = df.pivot(
    values="test_accuracy", index="dataset", columns="model", aggregate_function="mean"
).drop_nulls()
stats_pivot

In [ ]:
models = df["model"].unique().to_list()
P = stats_pivot.select(models).to_numpy()

In [ ]:
from aeon.visualisation import plot_critical_difference

plot_critical_difference(P, models)